# Day 4: Documentation Report

## Task Completed
- **Objective**: Implement a RAG system using GROQ for inference and local PDF files as the knowledge base.
- **Data Sources**: `TheSealedNectar-Alhamdulillah-library.blogspot.in.pdf` and `Seerat e Mustafa_new.pdf`.
- **Tech Stack**: LangChain, ChromaDB, HuggingFace Embeddings, and GROQ API (Llama 3).

## Summary of Findings
(The RAG output provides the specific content insights generated from the documents.)

In [1]:
!pip install -U -q langchain langchain-groq langchain-community langchain-huggingface sentence-transformers chromadb pypdf

In [2]:
import os
import time
from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor
from google.colab import userdata
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings

# Note: Using langchain_chroma for more stable vector store operations
try:
    from langchain_chroma import Chroma
except ImportError:
    !pip install -q langchain-chroma
    from langchain_chroma import Chroma

from langchain_groq import ChatGroq

# Configure GROQ
os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')

# 1. Load reference documents
file_paths = [
    '/content/TheSealedNectar-Alhamdulillah-library.blogspot.in.pdf',
    '/content/Seerat e Mustafa_new.pdf'
]
docs = []
print("Loading PDFs...")
for path in tqdm(file_paths, desc="Loading files"):
    loader = PyPDFLoader(path)
    docs.extend(loader.load())

# 2. Optimized splitting
text_splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=80)
splits = text_splitter.split_documents(docs)
print(f"Created {len(splits)} chunks.")

# 3. Initialize Embeddings
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# 4. Create vectorstore with Parallel Indexing
print("Generating Embeddings & Creating Vector Store (Parallelized)...")
batch_size = 50
num_workers = 4
vectorstore = Chroma(embedding_function=embeddings)

def process_batch(i):
    batch = splits[i:i + batch_size]
    vectorstore.add_documents(batch)
    return len(batch)

indices = range(0, len(splits), batch_size)
with ThreadPoolExecutor(max_workers=num_workers) as executor:
    list(tqdm(executor.map(process_batch, indices), total=len(indices), desc="Parallel Indexing"))

# 5. Initialize LLM via GROQ
llm = ChatGroq(model_name="llama-3.1-8b-instant", temperature=0)

print("\nRAG System Ready.")

/tmp/ipykernel_28057/948741300.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Loading PDFs...


Loading files:   0%|          | 0/2 [00:00<?, ?it/s]

Created 2468 chunks.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Generating Embeddings & Creating Vector Store (Parallelized)...


Parallel Indexing:   0%|          | 0/50 [00:00<?, ?it/s]


RAG System Ready.


In [3]:
import pandas as pd
from tqdm.auto import tqdm

# Verified 25 questions based on your documents (The Sealed Nectar and Seerat e Mustafa)
final_test_set = [
    "What was the name of the emperor to whom Hadhrat 'Amr bin Umayyah Damiri handed the letter from Rasulullah?",
    "What were the two white-clothed men doing to Rasulullah's milk-brother when they were seen by the milk-brother?",
    "What was the name of the delegation mentioned in the 1st point of the 10th year A.H.?",
    "What was the specific instruction given by Prophet Muhammad (PBUH) to the public regarding fasting during a certain time period?",
    "What is the name of the book being concluded by the author?",
    "What was the name of the person who requested a loan from K'ab?",
    "What was the name of the bin Hishaam mentioned as one of Prophet Muhammad's (PBUH) enemies?",
    "What was the name of the person who threatened the camel so terrifyingly that it suffered a miscarriage?",
    "What was the number of the verse that the monk exclaimed about Prophet Muhammad (PBUH) being the leader of the worlds?",
    "What was the name of the man who, according to Haatib, professed to be the highest lord and was punished by Allah?",
    "What was the month in which Rasulullah (PBUH) sent 'Amr bin 'Aas with a letter to the two sons of Julandi?",
    "What was the number of disbelievers that the Prophet Muhammad (PBUH) and his followers managed to kill with Allah's permission before losing their courage and disobeying?",
    "What was the number of soldiers in the fully equipped army prepared by Heraclius?",
    "What was the name that Haashim's actual name was said to be by Imaam Maalik and Imaam Shaafi'ee?",
    "What was the object that was divinely commanded by Allah Ta'ala to halt wherever it was directed to?",
    "What was the specific instruction given to 'Abdullah bin Jahsh in the letter from Prophet Muhammad (PBUH) that he was not to open until two days into his journey?",
    "What was the number of the people (hypocrites) who told the believers to fear the huge army gathered against them?",
    "What was the name of the tribe that accepted Islam according to the text?",
    "What was the specific reason given for the Prophet's heart to be strengthened according to the verse cited?",
    "What was the name of the Sahabi who was sent to Najran to collect Jizya and Zakat?",
    "Describe the exact physical reaction of the Prophet (PBUH) during the first revelation according to the text.",
    "What purpose did Zaid accomplish according to the specific wording in Surah Al-Ahzab mentioned in the text?",
    "What was the reason for the rapid and profound changes in the Arabian Peninsula according to the concluding remarks?",
    "What name did 'Abdul Muttalib decide to give the Prophet (PBUH) against the tradition of his forefathers?",
    "What is the specific act that Prophet Muhammad (PBUH) prohibited the people from doing while offering Salaah?"
]

def get_rag_response(question):
    docs = vectorstore.similarity_search(question, k=3)
    context = "\n".join([d.page_content for d in docs])
    prompt = f"""Answer the question using ONLY the context provided below.
    Context: {context}
    Question: {question}
    Answer:"""
    return llm.invoke(prompt).content

results = []
for i, question in enumerate(tqdm(final_test_set, desc="Evaluating RAG vs LLM")):
    try:
        llm_resp = llm.invoke(question).content
    except:
        llm_resp = "Inference Error"
    try:
        rag_resp = get_rag_response(question)
    except:
        rag_resp = "Retrieval Error"
    results.append({"#": i + 1, "Question": question, "LLM Only Answer": llm_resp, "RAG Answer": rag_resp})

df_results = pd.DataFrame(results)
pd.set_option('display.max_colwidth', 250)
display(df_results)

Evaluating RAG vs LLM:   0%|          | 0/25 [00:00<?, ?it/s]

,#,Question,LLM Only Answer,RAG Answer
0,1,What was the name of the emperor to whom Hadhrat 'Amr bin Umayyah Damiri handed the letter from Rasulullah?,I'm not aware of any information about an emperor receiving a letter from Rasulullah (the Prophet Muhammad) via Hadhrat 'Amr bin Umayyah Damiri.,"The emperor to whom Hadhrat 'Amr bin 'Aas handed the letter from Rasulullah was not the emperor of Rome, but rather the two sons of Julandi; 'Abd and Jayfar."
1,2,What were the two white-clothed men doing to Rasulullah's milk-brother when they were seen by the milk-brother?,"I'm not aware of any specific information about two white-clothed men being seen by Rasulullah's milk-brother. However, I can provide some context. \n\nIn Islamic history, there is a story about two men who were seen by Hamza ibn Abdul Muttalib, ...",They were busy stitching him up after slitting open his chest.
2,3,What was the name of the delegation mentioned in the 1st point of the 10th year A.H.?,"I'm not aware of any information about a specific delegation mentioned in the 1st point of the 10th year A.H. (After Hijra). The Hijri calendar is used to measure time in the Islamic calendar, and it starts from the year of the Prophet Muhammad's...",The delegation of Banu Haaris.
3,4,What was the specific instruction given by Prophet Muhammad (PBUH) to the public regarding fasting during a certain time period?,"You're likely referring to the instruction given by Prophet Muhammad (PBUH) regarding fasting during the month of Ramadan. However, I think you might be asking about a specific time period, which could be referring to the month of Shawwal.\n\nThe...",He instructed a man to publicly announce: 'He who has not eaten as yet should fast whilst he who has already eaten should also abstain from food and drink like the fasting ones.'
4,5,What is the name of the book being concluded by the author?,"I'm not aware of any specific information about a book being concluded by an author. This conversation has just started, and I don't have any context about a book or an author. If you could provide more information or clarify your question, I'll ...","Unfortunately, the name of the book being concluded by the author is not explicitly mentioned in the provided context."
5,6,What was the name of the person who requested a loan from K'ab?,I'm not aware of any information about a person requesting a loan from K'ab. Could you provide more context or information about K'ab and the situation you're referring to? I'll do my best to help.,Muhammad bin Maslamah.
6,7,What was the name of the bin Hishaam mentioned as one of Prophet Muhammad's (PBUH) enemies?,I am unable to verify the name of the bin Hishaam mentioned as one of Prophet Muhammad's (PBUH) enemies.,Abu Jahal bin Hishaam.
7,8,What was the name of the person who threatened the camel so terrifyingly that it suffered a miscarriage?,I'm not aware of any specific information about a person threatening a camel in such a manner that it suffered a miscarriage. It's possible that you may be thinking of a fictional story or anecdote. If you could provide more context or clarify th...,There is no information in the provided context about a person threatening a camel so terrifyingly that it suffered a miscarriage.
8,9,What was the number of the verse that the monk exclaimed about Prophet Muhammad (PBUH) being the leader of the worlds?,I'm not aware of any specific verse where a monk exclaimed about Prophet Muhammad (PBUH) being the leader of the worlds. It's possible that you may be thinking of a different context or a hadith (Prophetic tradition) that is not widely known.\n\n...,25
9,10,"What was the name of the man who, according to Haatib, professed to be the highest lord and was punished by Allah?",I'm not aware of any information about a man who professed to be the highest lord and was punished by Allah according to Haatib.,Ghawrith bin Haaris.
